# 第0章 環境の確認・ロジアナの数値化・テレメトリの解析（ノートブック版）

第0章の Mac・Pi 側の Python のうち、**Pico・ロジックアナライザ・MQTT がなくても動く部分**を、合成データで試せるように1冊にまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。執筆時の Mac で約1分。PyMC のコンパイルとサンプリングに時間がかかります）。

| 節 | 試すこと | 教科書の手順 | 使うファイル |
|---|---|---|---|
| 1 | 解析環境の動作確認（import、MPS、CWT、statsmodels、PyMC） | 手順 A | `check_env.py` |
| 2 | ロジアナの記録から PWM の周波数とデューティ | 手順 E、0-3 | `la_freq.py` |
| 3 | SQLite のスナップショットから欠測率（Wilson 区間）と Pico の時計のずれ（ppm） | 手順 G、0-3（図0-3） | `read_db.py` |

- 0-6 の3つのファイルを `%%writefile` で書き出します。中身は本文と同じです。変えたのは、`read_db.py` の図の日本語フォントの候補を足したこと（`plt.rcParams["font.family"]` の行）だけです。
- **合成データ**：`synth_ch0.py`（本文にはないファイル）が、`sigrok-cli -O binary` と同じ形の `cap.bin`（4 MHz・1 秒、1サンプル1バイト、ビット0 = D0）と、`pi_mqtt_logger.py` と同じ表 `messages` を持つ SQLite のファイル（`lab_snapshot.db`）を作ります。真の値（PWM の周波数のずれ +20 ppm、Pico の時計のずれ +25・−12 ppm、欠けた seq）は `TRUTH` に入れ、確かめだけに使います。
- 各節では、ターミナルの `uv run python ○○.py …`（Pi では `~/labenv/bin/python la_freq.py …`）の代わりに、`run("○○.py …")` で同じ引数のまま実行し、`ok()` で本文の「結果の例」と照らし合わせます。
- **入れていないもの**：Mac・Pi への インストール（`setup_mac.sh`・`setup_pi.sh`）、Pico への書き込み（`pwm_test.py`・`pwm_test.c`・`pico_mqtt_pub.py`）、Pi の Mosquitto と `pi_mqtt_logger.py`、`sigrok-cli` での実物の記録、PulseView での表示。
- `check_env.py` は、Colab では MPS（Apple の GPU）がないので「MPS available: False」になり、行列積の時間は測りません。PyWavelets・pyserial・paho-mqtt が Colab に入っていなければ、1 節の最初のセルで入れます（入らなくても `check_env.py` は NG と表示して先へ進みます）。
- 乱数の種を変えなければ数値はほぼ同じですが、numpy・statsmodels の版で最後の桁が変わることがあります。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には numpy・scipy・pandas・matplotlib・statsmodels・torch がたいてい入っている。版は要確認）
import importlib, subprocess, sys
for mod, pkg in [("statsmodels", "statsmodels"), ("pywt", "PyWavelets"), ("pymc", "pymc"), ("serial", "pyserial"), ("paho.mqtt", "paho-mqtt")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
import statsmodels, numpy
print("OK statsmodels", statsmodels.__version__, "numpy", numpy.__version__)

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

In [ ]:
%%writefile synth_ch0.py
"""第0章のノートブック版：ロジアナの記録と、Pi 5 の SQLite のスナップショットの代わりを作る（本文にはないファイル）"""
import json
import sqlite3

import numpy as np

TRUTH = {}


def la_bin(path, fs=4_000_000, seconds=1.0, f_nom=1000.0, ppm=+20.0, duty=0.25, seed=1):
    """sigrok-cli -O binary と同じ形（1サンプル1バイト、ビット0 = D0）。Pico の PWM（GP15、1 kHz・25%）のつもり"""
    rng = np.random.default_rng(seed)
    n = int(fs * seconds)
    f = f_nom * (1 + ppm * 1e-6)        # Pico とロジアナの時計の差（両方の誤差をまとめたもの）
    t = np.arange(n) / fs + rng.uniform(0, 1 / f)
    d0 = ((t * f) % 1.0) < duty
    d1 = (np.arange(n) // 999) % 2      # ビット1 = D1 には別の信号（D0 だけを取り出せるか）
    (d0.astype(np.uint8) | (d1 << 1).astype(np.uint8)).tofile(path)
    TRUTH["la"] = dict(f=f, ppm=ppm, duty=duty)


def make_db(path, devices, period_s=5.0, t_start=1.7e9, seed=2):
    """pi_mqtt_logger.py と同じ表 messages を作る。
    devices: {dev: dict(minutes=…, lost=[(区間, seq), …], reboot_after=区間の長さの list, ppm=…)}"""
    rng = np.random.default_rng(seed)
    con = sqlite3.connect(path)
    con.execute("""CREATE TABLE IF NOT EXISTS messages(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        recv_unix REAL NOT NULL,
        topic TEXT NOT NULL,
        payload TEXT NOT NULL,
        dev TEXT,
        seq INTEGER)""")
    rows = []
    for dev, p in devices.items():
        t = t_start
        for k, n_seg in enumerate(p["segments"]):
            ticks0 = int(rng.uniform(3000, 9000))                 # 起動してから最初の送信までの ticks_ms
            for seq in range(n_seg):
                t_pico = seq * period_s                              # Pico の時計で数えた時刻 [s]
                t_true = t_pico / (1 + p["ppm"] * 1e-6)              # Pico の時計が ppm だけ速い
                recv = t + t_true + rng.exponential(0.02)            # Wi-Fi・ブローカの遅れ（平均 20 ms）
                if (k, seq) in p.get("lost", []):
                    continue
                msg = {"dev": dev, "seq": seq, "ticks_ms": ticks0 + int(round(t_pico * 1000)),
                       "temp_c": round(27.0 + 2.0 * seq / max(n_seg, 1) + rng.normal(0, 0.6), 2)}
                rows.append((recv, f"lab/{dev}/telemetry", json.dumps(msg), dev, seq))
            t += n_seg * period_s + 20.0                             # 再起動に 20 秒かかったとする
        TRUTH[dev] = p
    rows.sort()
    rows.insert(len(rows) // 2, (t_start + 1.0, "lab/debug", "hello", None, None))   # dev のない行（読み飛ばされる）
    con.executemany("INSERT INTO messages(recv_unix, topic, payload, dev, seq) VALUES (?,?,?,?,?)", rows)
    con.commit()
    con.close()

In [ ]:
import glob, json, os, re, shlex, subprocess, sys
import numpy as np
import pandas as pd
from IPython.display import Image, display

HOME = os.getcwd()   # このノートブックのフォルダ（教科書の ~/home-lab の代わり）


def run(args, cwd="."):
    """ターミナルの「uv run python ○○.py …」の代わり。cwd のフォルダで実行し、表示を文字列でも返す"""
    wd = os.path.join(HOME, cwd)
    words = []
    for w in shlex.split(args):   # data/bounce_*.csv のような * を、ターミナルと同じく名前の順に広げる
        hits = sorted(glob.glob(os.path.join(wd, w))) if any(c in w for c in "*?[") else []
        words += [os.path.relpath(p, wd) for p in hits] or [w]
    p = subprocess.run([sys.executable] + words, capture_output=True, text=True, cwd=wd)
    print(p.stdout, end="")
    keep = [l for l in p.stderr.splitlines() if "findfont" not in l and l.strip()]   # 候補のうち見つからないフォントの警告は出さない（日本語のフォントがあるかは、0-1 のセルの表示で分かる）
    if keep:
        print("\n".join(keep), file=sys.stderr)
    return p.stdout, p.returncode


def show(*paths, width=800):
    """保存された図を表示する"""
    for f in paths:
        print(f)
        display(Image(filename=os.path.join(HOME, f), width=width))


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def num(pattern, text, i=1):
    """表示（text）から、pattern の (…) の数を取り出す"""
    m = re.search(pattern, text)
    assert m, "表示に見つからない：" + pattern
    return float(m.group(i).replace("−", "-").replace("+", ""))


print("準備ができました")


import synth_ch0 as syn

## 1. 解析環境の動作確認（手順 A、`check_env.py`）

本文では `cd ~/home-lab && uv run python ch0/check_env.py` です。PyMC の小さなサンプリング（2 チェーン × 300 回）があるので、初回は数十秒かかります。

In [ ]:
%%writefile check_env.py
#!/usr/bin/env python3
"""第0章：Macの解析環境の動作確認
使い方: cd ~/home-lab && uv run python ch0/check_env.py
"""
import importlib
import platform
import sys
import time

import numpy as np


def check_imports():
    print(f"Python {sys.version.split()[0]}  CPU={platform.machine()}")
    if platform.machine() != "arm64":
        print("⚠ arm64 ではありません。Rosetta 経由の Python かもしれません（MPSが使えない原因になる）")
    names = [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"),
             ("matplotlib", "matplotlib"), ("librosa", "librosa"),
             ("PyWavelets", "pywt"), ("statsmodels", "statsmodels"),
             ("scikit-learn", "sklearn"), ("PyMC", "pymc"), ("ArviZ", "arviz"),
             ("PyTorch", "torch"), ("pyserial", "serial"), ("paho-mqtt", "paho.mqtt")]
    for label, mod in names:
        try:
            m = importlib.import_module(mod)
            print(f"  OK  {label:13s} {getattr(m, '__version__', '?')}")
        except Exception as e:  # 失敗しても他の確認は続ける
            print(f"  NG  {label:13s} {e}")


def check_mps():
    import torch
    ok = torch.backends.mps.is_available()
    print(f"MPS available: {ok}")
    if not ok:
        return
    x = torch.randn(2048, 2048, device="mps")
    y = x @ x                        # 1回目は MPS の準備（初期化）の時間が入るので、測らずに捨てる
    torch.mps.synchronize()
    t0 = time.perf_counter()
    for _ in range(10):
        y = x @ x
    torch.mps.synchronize()          # GPUの計算が終わるのを待ってから時間を測る
    dt = (time.perf_counter() - t0) / 10
    print(f"  2048x2048 行列積: {dt*1e3:.1f} ms/回（{2*2048**3/dt/1e9:.0f} GFLOP/s 程度）")


def check_cwt():
    import pywt
    fs = 1000.0
    t = np.arange(0, 1.0, 1 / fs)
    x = np.sin(2 * np.pi * 50 * t)                     # 50 Hz の正弦波
    scales = np.arange(2, 128)
    coef, freqs = pywt.cwt(x, scales, "morl", sampling_period=1 / fs)
    power = np.mean(np.abs(coef) ** 2, axis=1)
    print(f"CWT: パワー最大の周波数 = {freqs[np.argmax(power)]:.1f} Hz（50 Hz 付近ならOK）")


def check_stats():
    import statsmodels.api as sm
    rng = np.random.default_rng(1)
    xx = np.linspace(0, 1, 50)
    yy = 2.0 + 3.0 * xx + rng.normal(0, 0.1, xx.size)
    fit = sm.OLS(yy, sm.add_constant(xx)).fit()
    lo, hi = fit.conf_int()[1]
    print(f"statsmodels: 傾き = {fit.params[1]:.3f}（95%CI {lo:.3f}〜{hi:.3f}、真値 3）")


def check_pymc():
    import pymc as pm
    data = np.array([0.1, -0.3, 0.5, 0.2, 0.0])
    with pm.Model():
        mu = pm.Normal("mu", 0.0, 1.0)
        pm.Normal("y", mu, 1.0, observed=data)
        idata = pm.sample(300, tune=300, chains=2, cores=1, progressbar=False, random_seed=1)
    print(f"PyMC: mu の事後平均 = {float(idata.posterior['mu'].mean()):.3f}（解析解 ≈ {data.sum()/(len(data)+1):.3f}）")


def main():
    check_imports()
    check_mps()
    check_cwt()
    check_stats()
    check_pymc()
    print("すべて終了しました。")


if __name__ == "__main__":     # PyMCは並列処理を使うので、この書き方が必要
    main()

In [ ]:
out_env, rc = run("check_env.py")
ok(rc == 0 and "すべて終了しました。" in out_env, "check_env.py が最後まで終わった")
ok(abs(num(r"パワー最大の周波数 = ([\d.]+) Hz", out_env) - 50) < 3, "CWT のピークは約 50 Hz")
b = num(r"傾き = ([\d.]+)", out_env)
lo, hi = num(r"傾き = [\d.]+（95%CI ([\d.]+)〜", out_env), num(r"傾き = [\d.]+（95%CI [\d.]+〜([\d.]+)", out_env)
ok(abs(b - 3) < 0.2 and hi - lo < 0.3, f"statsmodels：傾き {b}（95%CI {lo}〜{hi}）が真値 3 に近い")
print("（注意：この 95%CI は真値 3 を含まない。95% の区間は 20 回に 1 回は外れるので、環境の故障ではない）" if not lo < 3 < hi else "")
mu, exact = num(r"mu の事後平均 = (-?[\d.]+)", out_env), num(r"解析解 ≈ (-?[\d.]+)", out_env)
ok(abs(mu - exact) < 0.1 and abs(exact - 0.5 / 6) < 1e-3, f"PyMC：mu の事後平均 {mu} が解析解 0.5/6 ≈ {exact} に近い")
ng = re.findall(r"NG +(\S+)", out_env)
print("NG の項目：", ng or "なし", "（Mac の本文の環境ではすべて OK になるはず。Colab では入っていないものが NG になることがある）")

## 2. ロジアナの記録から PWM の周波数とデューティ（手順 E、`la_freq.py`）

`pwm_test.py`（GP15 に 1 kHz・デューティ 25%）を、ロジアナで 4 MHz・1 秒記録したつもりの `cap.bin` を作ります。Pico とロジアナの時計の差は +20 ppm としました。まず 0-3 の数値例（平均電圧 3.3 V × 0.25、1サンプルの時間）を確かめます。

In [ ]:
print(f"平均 {3.3*0.25:.3f} V、1サンプル = {1e6/4e6:.2f} µs、1周期 = {4e6/1000:.0f} サンプル")
ok(abs(3.3 * 0.25 - 0.8) < 0.05, "デューティ 25% の平均電圧は 3.3 V × 0.25 ≈ 0.8 V")
ok(1e6 / 4e6 == 0.25, "4 MHz では 1 サンプル = 0.25 µs")
syn.la_bin("cap.bin")
print(syn.TRUTH["la"], os.path.getsize("cap.bin"), "バイト")

In [ ]:
%%writefile la_freq.py
#!/usr/bin/env python3
"""sigrok-cli の binary 出力から D0（CH0）の周波数とデューティを求める
使い方: ~/labenv/bin/python la_freq.py cap.bin 4000000
（8ch すべてを記録した場合、1サンプル=1バイトで、ビット0が D0）
"""
import sys
import numpy as np

path = sys.argv[1]
fs = float(sys.argv[2])                       # サンプリング周波数 [Hz]
raw = np.fromfile(path, dtype=np.uint8)
b = (raw & 0x01).astype(np.int8)              # ビット0 = D0
rise = np.flatnonzero((b[1:] == 1) & (b[:-1] == 0)) + 1   # 立ち上がりエッジの位置
if len(rise) < 3:
    sys.exit("立ち上がりエッジが少なすぎます。配線・GND・サンプルレートを確認してください")
T = np.diff(rise) / fs                        # 各周期 [s]
f = 1.0 / T.mean()
duty = b[rise[0]:rise[-1]].mean()             # 整数周期ぶんの High の割合
print(f"サンプル数 {len(b)}（{len(b)/fs:.3f} 秒）")
print(f"周波数   = {f:.4f} Hz（周期の標準偏差 {T.std()*1e6:.3f} µs、1サンプル = {1e6/fs:.3f} µs）")
print(f"デューティ = {duty*100:.3f} %")
print(f"公称1 kHzからのずれ = {(f/1000.0 - 1)*1e6:+.0f} ppm（サンプリング側の誤差も含む）")

In [ ]:
out_la, rc = run("la_freq.py cap.bin 4000000")
ok(rc == 0, "la_freq.py がエラーなく終わった")
f = num(r"周波数   = ([\d.]+) Hz", out_la)
ok(abs(f - syn.TRUTH["la"]["f"]) < 0.002 and abs(f - 1000) < 0.1, f"周波数 {f} Hz（1000.0x Hz。真の値 {syn.TRUTH['la']['f']:.4f} Hz）")
ok(abs(num(r"デューティ = ([\d.]+) %", out_la) - 25.0) < 0.05, "デューティ 25.0x %")
ok(num(r"周期の標準偏差 ([\d.]+) µs", out_la) <= 0.25, "周期の標準偏差は1サンプル（0.25 µs）程度以下")
ok(abs(num(r"ずれ = ([+-]\d+) ppm", out_la) - 20) <= 1, "公称 1 kHz からのずれ +20 ppm（合成の真の値）を、周期を約1000回平均して求められる")

## 3. テレメトリの欠測率と時計のずれ（手順 G、`read_db.py`）

`pi_mqtt_logger.py` と同じ表 `messages(id, recv_unix, topic, payload, dev, seq)` を持つ SQLite のファイルを作ります。`pico_mqtt_pub.py` と同じく 5 秒ごとに `{"dev", "seq", "ticks_ms", "temp_c"}` を送り、Pi が受け取るまでに平均 20 ms の遅れがあるとしました。

- まず、図0-3 と同じ小さな例：区間1（seq 0〜5）で1件、再起動したあとの区間2（seq 0〜4）で1件欠けると、$N = 6 + 5 = 11$、$k = 2$、$\hat p = 2/11 \approx 0.18$。
- 次に、30 分ぶん：`pico1` は 10 分のところで再起動し、2件欠ける。時計は +25 ppm 速い。`pico2` は 360 件とも届き、時計は −12 ppm。

In [ ]:
%%writefile read_db.py
#!/usr/bin/env python3
"""SQLiteスナップショットを読み、欠測率（Wilson区間）と Pico の時計のずれ（ppm）を推定する
使い方: uv run python read_db.py lab_snapshot.db
"""
import json
import sqlite3
import sys

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.proportion import proportion_confint

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False

path = sys.argv[1] if len(sys.argv) > 1 else "lab_snapshot.db"
con = sqlite3.connect(path)
df = pd.read_sql_query("SELECT recv_unix, topic, payload, dev, seq FROM messages "
                       "WHERE dev IS NOT NULL ORDER BY id", con)
if df.empty:
    sys.exit("データがありません。logger が動いているか、トピックが lab/... か確認してください")
p = df["payload"].apply(json.loads)
df["temp_c"] = p.apply(lambda d: d.get("temp_c"))
df["ticks_ms"] = p.apply(lambda d: d.get("ticks_ms"))
df["t"] = pd.to_datetime(df["recv_unix"], unit="s", utc=True).dt.tz_convert("Asia/Tokyo")

fig, ax = plt.subplots(figsize=(9, 3.5))
for dev, g in df.groupby("dev"):
    g = g.sort_values("recv_unix").reset_index(drop=True)
    # seq が小さくなったら Pico が再起動したとみなして区間を分ける
    seg = (g["seq"].diff() < 0).cumsum()
    expected, received = 0, 0
    for _, h in g.groupby(seg):
        expected += int(h["seq"].max() - h["seq"].min() + 1)
        received += int(h["seq"].nunique())
    lost = expected - received
    lo, hi = proportion_confint(lost, expected, alpha=0.05, method="wilson")
    print(f"[{dev}] 期待 {expected} 件・受信 {received} 件・欠測 {lost} 件")
    print(f"       欠測率 = {lost/expected*100:.3f} %（95% Wilson区間 {lo*100:.3f}〜{hi*100:.3f} %）")

    # 時計のずれ：最後の区間で ticks_ms[s] を受信時刻[s] に回帰。傾き-1 が相対誤差
    h = g[seg == seg.max()]
    if len(h) >= 20:
        x = (h["recv_unix"] - h["recv_unix"].iloc[0]).to_numpy()
        y = (h["ticks_ms"] / 1000.0).to_numpy()
        if np.all(np.diff(y) > 0):          # ticks_ms の折り返し（約12日）が無いときだけ
            fit = sm.OLS(y, sm.add_constant(x)).fit()
            b = fit.params[1]
            blo, bhi = fit.conf_int()[1]
            print(f"       Picoの時計のずれ = {(b-1)*1e6:+.1f} ppm（95%CI {(blo-1)*1e6:+.1f}〜{(bhi-1)*1e6:+.1f}）"
                  f"  ※記録時間 {x[-1]/3600:.2f} 時間")
        else:
            print("       ticks_ms が折り返しているため時計の解析は省略")
    ax.plot(g["t"], g["temp_c"], ".", ms=3, label=dev)

ax.set_xlabel("時刻（JST）")
ax.set_ylabel("内蔵温度センサ [°C]（目安）")
ax.legend()
fig.tight_layout()
fig.savefig("telemetry.png", dpi=150)
print("telemetry.png を保存しました")

In [ ]:
for f_ in ("fig03.db", "lab_snapshot.db"):
    if os.path.exists(f_):
        os.remove(f_)
syn.make_db("fig03.db", {"pico1": dict(segments=[6, 5], lost=[(0, 2), (1, 3)], ppm=0.0)})
out_f, rc = run("read_db.py fig03.db")
ok(rc == 0 and "期待 11 件・受信 9 件・欠測 2 件" in out_f, "図0-3 の例：期待 11 件、欠測 2 件")
ok(abs(num(r"欠測率 = ([\d.]+) %", out_f) - 100 * 2 / 11) < 0.01, "p̂ = 2/11 ≈ 0.18")

In [ ]:
syn.make_db("lab_snapshot.db", {"pico1": dict(segments=[120, 240], lost=[(0, 57), (1, 130)], ppm=25.0),
                                "pico2": dict(segments=[360], lost=[], ppm=-12.0)})
out_db, rc = run("read_db.py lab_snapshot.db")
show("telemetry.png")

In [ ]:
# 確かめ（0-7 の read_db.py の結果の例）
ok(rc == 0, "read_db.py がエラーなく終わった（dev のない行は読み飛ばされる）")
b1, b2 = out_db.split("[pico2]")
ok("期待 360 件・受信 358 件・欠測 2 件" in b1, "pico1：再起動で区間を分けて、期待 360 件・欠測 2 件")
ok("期待 360 件・受信 360 件・欠測 0 件" in b2, "pico2：30 分（360 件）で欠測 0 件")
up0 = num(r"Wilson区間 0.000〜([\d.]+) %", b2)
ok(0.9 < up0 < 1.2, f"欠測 0 件でも上側の区間は 0% にならない（360 件なら上限 {up0}% ≈ 約1%）")
for blk, dev in ((b1, "pico1"), (b2, "pico2")):
    ppm = num(r"ずれ = ([+-][\d.]+) ppm", blk)
    lo, hi = num(r"95%CI ([+-][\d.]+)〜", blk), num(r"95%CI [+-][\d.]+〜([+-][\d.]+)）", blk)
    ok(lo < syn.TRUTH[dev]["ppm"] < hi and abs(ppm) < 100,
       f"{dev}：時計のずれ {ppm:+.1f} ppm（数十 ppm 以内）、95%CI {lo:+.1f}〜{hi:+.1f} が真の値 {syn.TRUTH[dev]['ppm']:+.0f} ppm を含む")

## おわりに

- `check_env.py` で解析環境を確かめ、`la_freq.py` で 1 サンプル 0.25 µs のロジアナでも周期を平均すれば ppm の桁まで周波数が求まること、`read_db.py` で再起動をまたいだ欠測率と、0 件でも上側の区間が 0% にならないこと、受信時刻への回帰で時計のずれが求まることを見ました。
- 実物では、ロジアナ・Pi・Pico のそれぞれの時計の誤差がまざります（`la_freq.py` の ppm は「サンプリング側の誤差も含む」、`read_db.py` は Pi 側の時刻のゆらぎも含む）。合成データではそれをひとつの差にまとめています。
- 自分の `cap.bin` や `lab_snapshot.db` を Colab の左のファイルの欄に上げれば、同じ `run(...)` で解析できるはずです（執筆時には確かめていません）。